# SpearShift: cloud run of the two-stage indicator detector (TASK_08, Colab GPU)

Runs `scripts/06_run_twostage.py extract` for the three models of `configs/twostage.yaml` on a Colab GPU
(`qwen25-7b` and `gemma2-9b` on `colab_t4`, `gemma2-27b` on `colab_l4`) and copies the rows back to Google
Drive after every cell. Exploratory, post hoc, supervisor-requested (`docs/QUESTIONS.md` item 18); the frozen
artefacts of the main study are never written.

`MODE`: `smoke` = the 20-item dev smoke sample (TASK_08 section 9 step 2; results/smoke_twostage/), `dev` = the
full dev extraction (796 items, step 3), `eval` = the evaluation items (5,207, step 4). Same Drive layout as
`cloud_run.ipynb` (`DRIVE_DIR/data/processed/...`). **Compute units (item 18j/k):** read "Available compute
units" (Runtime > View resources) before running cell 7 and enter it in `UNITS_BEFORE` (cell 1); when cell 7 has
finished, read it again, enter it in `UNITS_AFTER` of cell 9 and run cell 9: it records both readings, the sessions
they cover, the wall-clock estimate and the gap (`06_run_twostage.py units`), then writes and prints the budget
projection for plans A (dev + full eval) and B (dev + core eval) (`06_run_twostage.py budget`).

In [ ]:
#@title 1. Parameters
TAG = "main"                        #@param {type:"string"}
# TAG: a tag (v1.1.0-twostage once it exists) or a branch holding TASK_08 step 1 and later (main)
REPO_URL = "https://github.com/aliholmes5544/LLM.git"  #@param {type:"string"}
TIER = "colab_t4"                   #@param ["colab_t4", "colab_l4"]
MODELS = ""                         #@param {type:"string"}
MODE = "smoke"                      #@param ["smoke", "dev", "eval"]
DRIVE_DIR = "/content/drive/MyDrive/llm-thesis"  #@param {type:"string"}
UNITS_BEFORE = 0.0                  #@param {type:"number"}
REPO_DIR = "/content/llm"
# MODELS empty = the models of configs/twostage.yaml whose eval_tier equals TIER. UNITS_BEFORE = the Colab
# "Available compute units" reading taken just before cell 7 (0 = not entered); every session records it next to
# the wall-clock estimate. The reading after the run goes into cell 9.

In [ ]:
#@title 2. Mount Drive, check the GPU and the Drive layout
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], capture_output=True, text=True).stdout)
REQUIRED = ["data/processed/records.jsonl", "data/processed/partitions.json",
            "data/processed/variants_eval.jsonl", "data/processed/variants_dev.jsonl"]
missing = [p for p in REQUIRED if not os.path.exists(os.path.join(DRIVE_DIR, p))]
assert not missing, f"missing on Drive under {DRIVE_DIR}: {missing}"
for d in ("results/raw", "results/smoke_twostage", "reports", "logs"):
    os.makedirs(os.path.join(DRIVE_DIR, d), exist_ok=True)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
expected_gpu = {"colab_t4": "T4", "colab_l4": "L4"}[TIER]
assert expected_gpu in gpu, f"tier {TIER} expects a {expected_gpu} GPU, this runtime has {gpu!r} (Runtime > Change runtime type)"
print("GPU ok:", gpu)

In [ ]:
#@title 3. Clone the repository at the tag (model weights live outside the repo dir and survive a re-clone)
import os, subprocess
MODELS_DIR = "/content/models"
os.chdir("/content")
if os.path.exists(REPO_DIR):
    subprocess.run(["rm", "-rf", REPO_DIR], check=True)
subprocess.run(["git", "clone", "--quiet", "--branch", TAG, "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.symlink(MODELS_DIR, os.path.join(REPO_DIR, "models"))
os.chdir(REPO_DIR)
tag = subprocess.run(["git", "describe", "--tags", "--exact-match"], capture_output=True, text=True).stdout.strip()
assert tag == TAG or not TAG.startswith("v"), f"checked out {tag!r}, expected {TAG!r}"
assert os.path.exists("configs/twostage.yaml"), "this checkout predates TASK_08 step 1; choose a later TAG"
print("repository at", REPO_DIR, "tag", tag, "commit", subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 4. Install the pinned detection requirements with the CUDA build of llama-cpp-python 0.3.36 (+ scikit-learn for stage 2)
import os, re, subprocess, sys
nvcc = subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout
m = re.search(r"release (\d+)\.(\d+)", nvcc)
cuda_major = int(m.group(1)) if m else 12
index = ("https://abetlen.github.io/llama-cpp-python/whl/cu130" if cuda_major >= 13 else "https://abetlen.github.io/llama-cpp-python/whl/cu124")
pip = [sys.executable, "-m", "pip", "install", "-q"]
subprocess.run(pip + ["llama-cpp-python==0.3.36", "--prefer-binary", "--no-cache-dir", "--extra-index-url", index], check=True)
subprocess.run(pip + ["-r", "requirements-detect.txt", "scikit-learn==1.9.1", "joblib"], check=True)
import llama_cpp
assert llama_cpp.__version__ == "0.3.36", llama_cpp.__version__
if not llama_cpp.llama_supports_gpu_offload():
    subprocess.run(pip + ["--force-reinstall", "--no-cache-dir", "--no-binary", "llama-cpp-python", "llama-cpp-python==0.3.36"],
                   check=True, env={**os.environ, "CMAKE_ARGS": "-DGGML_CUDA=on"})
    raise SystemExit("built from source: Runtime > Restart session, then run the notebook again from cell 1")
print("llama-cpp-python", llama_cpp.__version__, "GPU offload", llama_cpp.llama_supports_gpu_offload())

In [ ]:
#@title 5. Pull the processed data from Drive, verify the fixed variants artefact, restore previous two-stage results
import hashlib, json, os, shutil, subprocess, time
os.makedirs("data/processed", exist_ok=True)
for p in REQUIRED:
    shutil.copy2(os.path.join(DRIVE_DIR, p), p)
fixed = json.load(open("reports/variants_eval_fixed_artifacts.json"))["files"]["data/processed/variants_eval.jsonl"]
h = hashlib.sha256(open("data/processed/variants_eval.jsonl", "rb").read()).hexdigest()
assert h == fixed["sha256"], f"variants_eval.jsonl sha256 {h} differs from the fixed artefact"
env = os.path.join(DRIVE_DIR, ".env")
if os.path.exists(env):
    shutil.copy2(env, ".env")
REPORT_FILES = ("twostage_report.json", "models_download_report.json")

def sync_to_drive(quiet=False):
    for d in ("results/raw", "results/smoke_twostage"):
        if os.path.isdir(d):
            for sub in sorted(os.listdir(d)):
                if sub.startswith("twostage_") or d.endswith("smoke_twostage"):
                    subprocess.run(["rsync", "-a", os.path.join(d, sub) + "/", os.path.join(DRIVE_DIR, d, sub) + "/"], check=True)
    for f in REPORT_FILES:
        if os.path.exists(os.path.join("reports", f)):
            shutil.copy2(os.path.join("reports", f), os.path.join(DRIVE_DIR, "reports", f))
    if not quiet:
        print(time.strftime("%H:%M:%S"), "synced two-stage results and reports to", DRIVE_DIR)

# restore ONLY the two-stage files (results/raw/twostage_*, results/smoke_twostage) so that every cell resumes;
# the main study's results/raw/<model>/ files are in git at the tag and are never written here
src = os.path.join(DRIVE_DIR, "results/raw")
if os.path.isdir(src):
    for sub in sorted(os.listdir(src)):
        if sub.startswith("twostage_"):
            os.makedirs(os.path.join("results/raw", sub), exist_ok=True)
            subprocess.run(["rsync", "-a", os.path.join(src, sub) + "/", os.path.join("results/raw", sub) + "/"], check=True)
src = os.path.join(DRIVE_DIR, "results/smoke_twostage")
if os.path.isdir(src):
    os.makedirs("results/smoke_twostage", exist_ok=True)
    subprocess.run(["rsync", "-a", src + "/", "results/smoke_twostage/"], check=True)
for f in REPORT_FILES:
    s = os.path.join(DRIVE_DIR, "reports", f)
    if os.path.exists(s):
        shutil.copy2(s, os.path.join("reports", f))
prev = subprocess.run(["bash", "-c", "find results/raw/twostage_* results/smoke_twostage -name '*.jsonl' 2>/dev/null"], capture_output=True, text=True).stdout.strip()
print("restored from Drive:\n" + (prev or "(no previous two-stage results)"))

In [ ]:
#@title 6. Download the GGUF files of the selected models and verify their sha256
import subprocess, sys, yaml
mcfg, tcfg = yaml.safe_load(open("configs/models.yaml")), yaml.safe_load(open("configs/twostage.yaml"))
MODEL_KEYS = [k.strip() for k in MODELS.split(",") if k.strip()] or [k for k in tcfg["models"] if mcfg["models"][k]["eval_tier"] == TIER]
bad = [k for k in MODEL_KEYS if k not in tcfg["models"]]
assert not bad, f"{bad} not in configs/twostage.yaml models (three, no more)"
print("models:", MODEL_KEYS)
subprocess.run([sys.executable, "scripts/06_download_models.py", "--tier", TIER, "--models", ",".join(MODEL_KEYS)], check=True)
sync_to_drive()

In [ ]:
#@title 7. Extraction: smoke (20 dev items), dev (796) or eval (5,207) per model; synced to Drive every 10 min and after every cell
import datetime, os, subprocess, sys, threading, time
RUN_T0 = time.time()                 # start of the span the compute-unit reading covers (cell 9)
units = [] if not UNITS_BEFORE else ["--units-before", str(UNITS_BEFORE)]
print("units before:", UNITS_BEFORE or "(not entered)", "| when this cell has finished, read the units again and run cell 9")
for m in MODEL_KEYS:
    args = [sys.executable, "-u", "scripts/06_run_twostage.py", "extract", "--model", m, "--tier", TIER, "--backend", "cuda"] + units
    args += ["--smoke", "--partition", "dev"] if MODE == "smoke" else ["--partition", MODE]
    log_path = os.path.join(DRIVE_DIR, "logs", f"twostage_{m}_{MODE}.log")
    print(f"=== {datetime.datetime.now(datetime.timezone.utc):%Y-%m-%d %H:%M:%S}Z twostage {m} {MODE}", flush=True)
    stop = threading.Event()
    ticker = threading.Thread(target=lambda: [sync_to_drive(quiet=True) for _ in iter(lambda: not stop.wait(600), False)], daemon=True)
    ticker.start()
    with open(log_path, "a") as lf:
        p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end="", flush=True)
            lf.write(line)
        rc = p.wait()
    stop.set(); ticker.join(timeout=5)
    sync_to_drive()
    if rc != 0:
        raise SystemExit(f"twostage {m} {MODE} exited with code {rc}; see {log_path}; fix and run the notebook again (it resumes)")

In [ ]:
#@title 8. Summary of the two-stage cells (reports/twostage_report.json) and the compute-unit record
import json, os
path = "reports/twostage_report.json"
if os.path.exists(path):
    rep = json.load(open(path))
    print(f"{'cell':34s} {'done/expected':>14s} {'parse fail':>10s} {'volunteered':>11s} {'median s':>9s} {'out tok':>7s}  gpu")
    for k, c in sorted(rep["cells"].items()):
        print(f"{k:34s} {c['n_done']:>6d}/{c['n_expected']:<7d} {c.get('parse_failure_rate') or 0:>10.1%} {c.get('volunteered_verdict_rate') or 0:>11.1%} "
              f"{c['latency_median_s']:>9.2f} {c.get('output_tokens_mean') or 0:>7.1f}  {c.get('gpu')}")
        for s in c.get("sessions", [])[-1:]:
            print("    units:", s.get("units"))
print(f"Next: enter the units reading in cell 9 and run it; then copy {DRIVE_DIR}/results/raw/twostage_*, {DRIVE_DIR}/results/smoke_twostage and {DRIVE_DIR}/reports/twostage_report.json into the repository on the laptop and commit them.")

In [ ]:
#@title 9. Record the compute-unit reading after the run and project the budget (item 18 j/k; plans A and B)
UNITS_AFTER = 0.0                   #@param {type:"number"}
import subprocess, sys, time
assert UNITS_BEFORE and UNITS_AFTER, "enter UNITS_BEFORE (cell 1, read before cell 7) and UNITS_AFTER (read after cell 7) first"
assert UNITS_AFTER < UNITS_BEFORE, "the after reading must be smaller than the before reading"
span = round(time.time() - RUN_T0, 1)
args = [sys.executable, "scripts/06_run_twostage.py", "units", "--tier", TIER, "--models", ",".join(MODEL_KEYS),
        "--units-before", str(UNITS_BEFORE), "--units-after", str(UNITS_AFTER), "--span-s", str(span)]
args += ["--smoke"] if MODE == "smoke" else ["--partition", MODE]
subprocess.run(args, check=True)
subprocess.run([sys.executable, "scripts/06_run_twostage.py", "budget"], check=True)
sync_to_drive()
print(f"Next: copy {DRIVE_DIR}/results/smoke_twostage (and results/raw/twostage_* after dev or eval) and {DRIVE_DIR}/reports/twostage_report.json into the repository on the laptop and commit them.")